In [118]:
import rasterio as rio
import numpy as np
import matplotlib.pyplot as plt
import os
import geopandas as gpd
from pathlib import Path

In [119]:
import shutil
from itertools import combinations
from pathlib import Path
import geopandas as gpd
from shapely.ops import unary_union

SRC = Path('/Users/wm722/Documents/SCDs/results/UK/Raw')
OUT = Path('/Users/wm722/Documents/SCDs/results/UK/Cleaned')
TILE_COL = 'bng_ref'  # whichever column in the index holds names like SY78sw

index = gpd.read_file('/Users/wm722/Documents/SCDs/data/UK/Index.gpkg')
geom = dict(zip(index[TILE_COL].str.lower(), index.geometry))
key = lambda t: t.split('_')[0].lower()

# tiles per run
runs = {}
for f in sorted(SRC.rglob('covered_bounds.txt')):
    line = next(l for l in f.read_text().splitlines() if l.startswith('Tiles used:'))
    runs[f.parent] = [t.strip() for t in line.split(':', 1)[1].split(',') if t.strip()]

cov = {d: unary_union([geom[key(t)] for t in ts]) for d, ts in runs.items()}
remove = {d: [] for d in runs}

# always trim the second of each pair, using coverage as already trimmed
for a, b in combinations(runs, 2):
    inter = cov[a].intersection(cov[b])
    if inter.area < 1:
        continue
    runs[b] = [t for t in runs[b] if geom[key(t)].intersection(inter).area < 1]
    cov[b] = unary_union([geom[key(t)] for t in runs[b]])
    remove[b].append(inter)
    print(f'{a.name} x {b.name}: trimmed {b.name}, {len(runs[b])} tiles left')

# copy everything, then fix up the trimmed runs
shutil.copytree(SRC, OUT, dirs_exist_ok=True)

for d in runs:
    if not remove[d]:
        continue
    dest = OUT / d.relative_to(SRC)
    mask = gpd.GeoDataFrame(geometry=[unary_union(remove[d])], crs=index.crs)

    for shp in dest.glob('*.shp'):
        g = gpd.read_file(shp)
        g['geometry'] = g.geometry.make_valid()
        g = gpd.overlay(g, mask.to_crs(g.crs), how='difference')
        g = g[~g.is_empty]
        g.to_file(shp)
        print(f'{dest.name}/{shp.name}: {len(g)} features')

    txt = dest / 'covered_bounds.txt'
    lines = [f"Tiles used: {', '.join(runs[d])}" if l.startswith('Tiles used:') else l
             for l in txt.read_text().splitlines()]
    txt.write_text('\n'.join(lines) + '\n')

Beverely x SE50-75: trimmed SE50-75, 79 tiles left
Beverely x SE76-99: trimmed SE76-99, 70 tiles left
Beverely x TAMixedv2: trimmed TAMixedv2, 8 tiles left
CornwallSW_Final_Fixed_Final2 x SW: trimmed SW, 94 tiles left
Darlington x Windermere: trimmed Windermere, 74 tiles left
Driffield x SE50-75: trimmed SE50-75, 75 tiles left
Driffield x SE76-99: trimmed SE76-99, 26 tiles left
Grassington x SE00-25: trimmed SE00-25, 72 tiles left
Keswick x Windermere: trimmed Windermere, 36 tiles left
Liverpool_v3 x Preston: trimmed Preston, 81 tiles left
NYM x SE00-25: trimmed SE00-25, 64 tiles left
NYM x SE26-49v2: trimmed SE26-49v2, 82 tiles left
NYM x SE50-75: trimmed SE50-75, 67 tiles left
NYM x SE76-99: trimmed SE76-99, 20 tiles left
SS00-49 x SXMixed: trimmed SXMixed, 35 tiles left
SS50-99 x SXMixed: trimmed SXMixed, 34 tiles left
STSY00-39v2 x SW: trimmed SW, 93 tiles left
STSY00-39v2 x WellsFull: trimmed WellsFull, 76 tiles left
STSY40-59 x WellsFull: trimmed WellsFull, 16 tiles left
SU26-49 

In [120]:
results_dirs = ['/Users/wm722/Documents/SCDs/results/UK/Cleaned']

covered_tiles = set()
duplicate_tiles = set()
index = gpd.read_file('/Users/wm722/Documents/SCDs/data/UK/Index.gpkg')

for results_dir in results_dirs:
    for root, dirs, files in os.walk(results_dir):
        for file in files:
            if file.endswith('.txt'):
                file_path = os.path.join(root, file)
                with open(file_path, 'r') as f:
                    content = f.read()
                    print(f.name)
                    tiles = list(content.split('Tiles used: ')[1].split('\n'))[0].split(', ')
                    for tile in tiles:
                        if tile in covered_tiles:
                            duplicate_tiles.add(tile)
                        else:
                            covered_tiles.add(tile)


covered_bounds = []

names = {t.split('_')[0].upper() for t in covered_tiles}
s = index.loc[index['bng_ref'].isin(names), 'geometry']

duplicate_names = {t.split('_')[0].upper() for t in duplicate_tiles}
duplicates = index.loc[index['bng_ref'].isin(duplicate_names), 'geometry']

/Users/wm722/Documents/SCDs/results/UK/Cleaned/SW/covered_bounds.txt
/Users/wm722/Documents/SCDs/results/UK/Cleaned/TL61-83/covered_bounds.txt
/Users/wm722/Documents/SCDs/results/UK/Cleaned/TQ76-99/covered_bounds.txt
/Users/wm722/Documents/SCDs/results/UK/Cleaned/STSY00-39v2/covered_bounds.txt
/Users/wm722/Documents/SCDs/results/UK/Cleaned/SX/covered_bounds.txt
/Users/wm722/Documents/SCDs/results/UK/Cleaned/SK00-25/covered_bounds.txt
/Users/wm722/Documents/SCDs/results/UK/Cleaned/TAMixedv2/covered_bounds.txt
/Users/wm722/Documents/SCDs/results/UK/Cleaned/Grassington/covered_bounds.txt
/Users/wm722/Documents/SCDs/results/UK/Cleaned/SS00-49/covered_bounds.txt
/Users/wm722/Documents/SCDs/results/UK/Cleaned/Wooley/covered_bounds.txt
/Users/wm722/Documents/SCDs/results/UK/Cleaned/TQ50-75/covered_bounds.txt
/Users/wm722/Documents/SCDs/results/UK/Cleaned/Wareham/covered_bounds.txt
/Users/wm722/Documents/SCDs/results/UK/Cleaned/SP69-91/covered_bounds.txt
/Users/wm722/Documents/SCDs/results/UK/

In [121]:
covered = gpd.GeoDataFrame(geometry=s, crs=index.crs)
covered.to_file('~/Documents/SCDs/results/UK/QGIS/covered_tiles.gpkg', driver='GPKG')

dups = gpd.GeoDataFrame(geometry=duplicates, crs=index.crs)
dups.to_file('~/Documents/SCDs/results/UK/QGIS/duplicate_tiles.gpkg', driver='GPKG')

In [122]:
shp_out = '/Users/wm722/Documents/SCDs/results/UK/FlatForQ'
results_dir = '/Users/wm722/Documents/SCDs/results/UK/Cleaned/'

for root, dirs, files in os.walk(results_dir):
    for directory in dirs:
        name = directory
        prediction_file = os.path.join(root, directory, 'predictions.shp')
        print(f'Processing {name}...')
        if os.path.exists(prediction_file):
            predictions = gpd.read_file(prediction_file)
            predictions.to_file(os.path.join(shp_out, f'{name}.shp'), driver='ESRI Shapefile')
        else:
            continue
            


Processing SW...
Processing TL61-83...
Processing TQ76-99...
Processing STSY00-39v2...
Processing SX...
Processing SK00-25...
Processing TAMixedv2...
Processing Grassington...
Processing SS00-49...
Processing Wooley...
Processing TQ50-75...
Processing Wareham...
Processing SP69-91...
Processing Nailsworth...
Processing TL15-37...
Processing Beverely...
Processing TQMixed...
Processing TF00-25...
Processing TF50-99v3...
Processing SE76-99...
Processing Warminster...
Processing CornwallSW_Final_Fixed_Final2...
Processing TQ26-49v2...
Processing WellsFull...
Processing Ludlow...
Processing SS50-99...
Processing Carlisle...
Processing SU00-25...
Processing SE50-75...
Processing TQ00-25...
Processing SXMixed...
Processing SP92-TL14...
Processing SZ...
Processing Driffield...
Processing TL38-60v2...
Processing Preston...
Processing TF26-49...
Processing Crewe...
Processing SK76-99...
Processing Morpeth...
Processing Birm...
Processing STSY40-59...
Processing Keswick...
Processing SP23-45...
